# Training a BIT model on a dense motor target

[04_bit_pipeline](04_bit_pipeline.ipynb) trains BIT to emit characters. This notebook trains the same model to predict a two-dimensional cursor signal on every time bin. The trunk is the same `BitCrossEncoder`. What changes is the head, which outputs the target's channels instead of character classes when the bundle declares a `target_stream`, and the loss, a masked mean squared error in place of CTC. The data goes through `BitDataBundle.from_collection`, the one-call path at the end of 04.

It runs on three cursor datasets, `karpowicz_2025` (participant T11), `wilson_2025` (participant T5) and `deo_2024` (also T5). `deo_2024` ran no online decoder in its open-loop blocks and publishes no decoded velocity, so it joins the intention target only. Its open-loop blocks are why it is here: the cursor followed a computer-generated trajectory, so no decoder shaped what T5 saw. `singer_2025` also publishes `DecodedCursorVelocitySeries`, but it bins at 100 Hz against 50 Hz for the others, and the bundle refuses a mixed rate until resampling is wired up.

It compares two targets on the same trials of `karpowicz_2025` and `wilson_2025`, then trains the intention target again with `deo_2024` added:

| Target | Built from | What it is |
| --- | --- | --- |
| velocity | `DecodedCursorVelocitySeries` | The velocity each dataset's online decoder commanded. This is the online decoder's output, not a measured movement: the participants are paralyzed, and the cursor moved because a decoder moved it. |
| intention | `CursorIntentionSeries`, derived from `CursorPositionSeries` and `TargetPositionSeries` | The unit vector from the cursor to the cued target on every bin. The cued target is set by the task, so this label does not come from the old decoder. |

> **Requires the data drive.** `NEURALSHED_DATA_ROOT` must point at a mount holding all three datasets. The first cell fails with a clear message if it does not.

In [ ]:
import os, logging, time, warnings
warnings.filterwarnings("ignore")
os.environ["TQDM_DISABLE"] = "1"     # neuralset's prepare() and Lightning's progress bars
logging.disable(logging.INFO)        # extractors log per-event at INFO

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv

from neuralshed.datasets.utils import dataset_dir

load_dotenv()                        # NEURALSHED_DATA_ROOT from the repo's .env
pd.set_option("display.width", 170, "display.max_columns", 40)

MEMBERS = ["karpowicz_2025", "wilson_2025", "deo_2024"]
VELOCITY_MEMBERS = ["karpowicz_2025", "wilson_2025"]   # deo_2024 publishes no velocity
QUERY = "timeline_index < 10"        # None runs every session
EPOCHS = 15
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

for member in MEMBERS:
    print(f"{member:16s} {dataset_dir(member, must_exist=True)}")
print("device:", DEVICE)

## 1. One bundle per target

`motor_cursor_collection()` is `CombineStudies`, `ValidateTarget` and `AssignHashSplit`, the speech collection's shape without the text steps. One trigger row is one cursor movement (`action_kind == 'cursor_control'`), and each trial is segmented at its own duration.

The two bundles differ only in their target. Velocity reads `DecodedCursorVelocitySeries` as each dataset published it. Intention reads `CursorIntentionSeries`, which no dataset publishes: `DeriveStream` computes it from the cursor and target positions and appends it as a stream, so by the time the bundle sees it, it is an ordinary stream with its own width, rate and layout keys. Deriving in the collection rather than at extraction is what makes the target visible in the events, the manifest and the preflight report, and it computes over the whole recording instead of inside one trial window. `QUERY` keeps the first ten timelines of each dataset so the notebook runs in a few minutes.

`deo_2024` also runs its own filters, from `MOTOR_CURSOR_MEMBER_QUERIES`. It keeps the right cursor, whose tuning is the strong one in these left-hemisphere arrays, and open-loop blocks only. It also drops left-hand-only trials, where the right cursor sits on its target and has no direction. The third bundle is the intention bundle with `deo_2024` added.

In [ ]:
from neuralshed.models import motor_cursor_collection, motor_cursor_identities
from neuralshed.models.bit import BitDataBundle


def intention_bundle(members):
    identities = motor_cursor_identities(members)
    return BitDataBundle.from_collection(
        motor_cursor_collection(members, query=QUERY, derive=("intention",)),
        inputs=("SpikeCounts", "BandPower"),
        targets=("CursorIntentionSeries",),
        target_stream="CursorIntentionSeries",
        regions=identities["regions"],
        participants=identities["participants"],
        already_normalized=identities["already_normalized"],
        verbose=False,
    )


identities = motor_cursor_identities(VELOCITY_MEMBERS)
velocity = BitDataBundle.from_collection(
    motor_cursor_collection(VELOCITY_MEMBERS, query=QUERY),
    inputs=("SpikeCounts", "BandPower"),
    targets=("DecodedCursorVelocitySeries",),
    target_stream="DecodedCursorVelocitySeries",
    regions=identities["regions"],
    participants=identities["participants"],
    already_normalized=identities["already_normalized"],
    verbose=False,
)
intention = intention_bundle(VELOCITY_MEMBERS)
intention_with_deo = intention_bundle(MEMBERS)
print(velocity)
print(intention)
print(intention_with_deo)

## 2. Same trials, different target

Both targets are read out of the same `Segmenter` window as the spikes, so they are aligned to them bin for bin. The two bundles hold the same trials in the same order and the same splits, which is what makes the comparison below paired. Adding `deo_2024` leaves the other two datasets' trials and splits unchanged, because the split hashes each trial's own timeline and start. Every intention vector has norm 1, except on bins where the cursor sits on the target, which have no direction and are left at zero.

In [ ]:
def trial_keys(bundle):
    return [(t.group_key, t.split, t.n_frames) for t in bundle.trials]


print("same trials, order and splits:", trial_keys(velocity) == trial_keys(intention))
shared = [t for t in intention_with_deo.trials if t.member in VELOCITY_MEMBERS]
print("same trials with deo_2024 added:", sorted(trial_keys(intention)) == sorted((t.group_key, t.split, t.n_frames) for t in shared))
print("deo_2024 trials:", sum(t.member == "deo_2024" for t in intention_with_deo.trials))

trial_v, trial_i = velocity.trials[0], intention.trials[0]
norms = np.linalg.norm(trial_i.target, axis=0)
print(f"intention norms on this trial: min {norms.min():.3f}, max {norms.max():.3f}")

pd.DataFrame([
    {"array": "SpikeCounts (input)", "shape": trial_i.arrays["SpikeCounts"].shape},
    {"array": "velocity target", "shape": trial_v.target.shape},
    {"array": "intention target", "shape": trial_i.target.shape},
]).set_index("array")

## 3. One model per target

`bundle.config()` passes the bundle's `target_stream` into `BitConfig`, which reads the target's width off the manifest. Both targets are two channels wide, so all three models have the same shape and differ only in what they are trained on. Each starts from the same seed.

In [ ]:
from neuralshed.models.bit import BitCrossEncoder

BUNDLES = {"velocity": velocity, "intention": intention, "intention with deo": intention_with_deo}


def fresh_model(bundle, seed=0):
    torch.manual_seed(seed)
    model = BitCrossEncoder(bundle.config())
    model.load_preprocessing(bundle.state)
    return model


pd.DataFrame([
    {
        "target": name,
        "target_stream": config.target_stream,
        "head_width": config.head_width,
        "regression": config.is_regression,
        "patch_bins": config.patch_bins,
    }
    for name, config in ((name, bundle.config()) for name, bundle in BUNDLES.items())
]).set_index("target")

## 4. Train and score each member

`train_regression` fits the model with a masked mean squared error on the patch-pooled target, so padded bins never reach the loss. `evaluate_by_member` reports R² (coefficient of determination) per dataset on the validation split, never pooled across datasets, since the velocity targets are in different units. The untrained score is the floor each run starts from.

Each bundle is trained with three seeds. On a subset this small a single run moves by a few hundredths, which is the same size as the differences we are looking at.

In [ ]:
from neuralshed.models.training import evaluate_by_member, train_regression

# Lightning's notebook progress bar captures the rest of the cell it runs in, so the results
# are shown in the next cell.
SEEDS = (0, 1, 2)
ACCELERATOR = "gpu" if DEVICE == "cuda" else "cpu"
rows, seconds = [], {}
for name, bundle in BUNDLES.items():
    start = time.time()
    for seed in SEEDS:
        model = fresh_model(bundle, seed)
        untrained = evaluate_by_member(model, bundle.val_loader(), device=DEVICE)
        train_regression(
            model,
            bundle.train_loader(seed=seed),
            bundle.val_loader(),
            max_epochs=EPOCHS,
            accelerator=ACCELERATOR,
        )
        trained = evaluate_by_member(model, bundle.val_loader(), device=DEVICE)
        for member in sorted(trained):
            rows.append({
                "member": member,
                "target": name,
                "seed": seed,
                "untrained R2": untrained.get(member, np.nan),
                "trained R2": trained.get(member, np.nan),
            })
    seconds[name] = time.time() - start

In [ ]:
for name, elapsed in seconds.items():
    print(f"{name}: {len(SEEDS)} runs of {EPOCHS} epochs in {elapsed:.0f} s")

runs = pd.DataFrame(rows)
(
    runs.groupby(["member", "target"])[["untrained R2", "trained R2"]]
    .agg(["mean", "std"])
    .round(3)
)

## 5. Reading the result

On this subset the two targets land close to each other, and which one is ahead depends on the dataset: velocity for `karpowicz_2025`, intention for `wilson_2025`.

`deo_2024` reaches an intention R² of about 0.80, as high as `wilson_2025`. Adding it leaves `karpowicz_2025` where it was (0.585 against 0.584) and lowers `wilson_2025` from 0.800 to 0.779, a drop larger than the spread across seeds. `deo_2024` and `wilson_2025` are both T5 and share one participant embedding.

On every session (`QUERY = None`, 69,438 trials, one seed) the picture changes:

| member | velocity | intention |
| --- | --- | --- |
| `karpowicz_2025` | 0.733 | 0.238 |
| `wilson_2025` | 0.851 | 0.663 |

Adding sessions raises the velocity score and lowers the intention score, and the validation set changes along with the training set. So the large gap on all sessions does not by itself show that the velocity score comes from relearning the old online decoder. It is just as consistent with the direction to the cued target being a poor intention label in some of the later sessions, for instance where the cursor holds on the target and the label is zero. Breaking the all-session scores down per session is the check that separates the two.

A direction and a velocity are also different quantities, so their R² values are not strictly comparable.